# Paso 5 — Analisis embeddings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap.umap_ as umap 
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from collections import defaultdict
import seaborn as sns
import os
import sys

## Analyse modules

In [ ]:
def analyze_modules(protein_module_map, go_terms_df, protein_metadata_df, go_metadata_df, all_proteins_list):
   # Obtener todos los IDs de módulos únicos, incluyendo -1 (ruid)
    unique_modules = np.unique(list(protein_module_map.values()))
    results = []

    protein_go_map = defaultdict(list)
    for _, row in go_terms_df.iterrows():
        protein_go_map[row['proteina']].append(row['GO_term'])
    
    go_term_details = go_metadata_df.set_index('GO_term')
    protein_deg_map = protein_metadata_df.set_index('proteina')['DEG'].to_dict()
    protein_target_group_map = protein_metadata_df.set_index('proteina')['Target_group'].to_dict()

    # Contar GO terms en todo el universo de proteinas con GO anotado
    all_go_terms_in_network = [term for prot_id in all_proteins_list if prot_id in protein_go_map for term in protein_go_map[prot_id]]
    global_go_counts = pd.Series(all_go_terms_in_network).value_counts().to_dict()
    
    # Total de proteínas con al menos un GO term asignado
    proteins_in_go_universe = len(set(p for p in all_proteins_list if protein_go_map[p])) # Total de proteínas con al menos un GO term asignado

    # Iterar sobre todos los módulos únicos, incluyendo -1
    for module_id in sorted(unique_modules):
        #if module_id == -1: 
        #    continue

        # Eliminamos la condición para que el clsuter de ruido sea procesado.


        # Proteinas del modulo, se filtran las prpoteinas que pertenecen al modulo actual
        module_proteins = [p for p, m in protein_module_map.items() if m == module_id]
        
        if not module_proteins:
            continue

        # 1. Enriquecimiento GO
        representative_go = "N/A (Ruido, no GO enriquecido)" if module_id == -1 else "N/A (No GO terms enriched)"
        representative_go_p_value = 1.0
        representative_combined_score = 0.0
        representative_go_z_score = 0.0

        if module_id != -1: # Realizar el test de enriquecimiento GO solo para clústeres válidos
            module_go_terms = [term for p in module_proteins for term in protein_go_map[p]]
            module_go_counts = pd.Series(module_go_terms).value_counts().to_dict()
            
            enriched_go_terms_list = []
            p_values = []
            z_scores = []

            # Test hipergeométrico para cada GO term en el módulo
            for go_term, module_count in module_go_counts.items():
                k = module_count
                M = global_go_counts.get(go_term, 0)  # Total de proteínas con este GO term en el universo
                n = len(module_proteins) # tamaño del módulo
                N = proteins_in_go_universe # Total de proteínas con al menos un GO term en el universo
                
                # Para evitar divisiones por cero o valores no validos 
                if N == 0 or M == 0 or n == 0:
                    p_val = 1.0 # significa que no hay enriquecimiento, no hay GO term en el universo o en el módulo, se asigna 1.0 porque no hay enriquecimiento
                    z_score = 0.0 # significa que no hay enriquecimiento, si es 0 es porque no hay GO term en el universo o en el módulo, los datos no son validos
                else:
                    # stats.hypergeom.sf(k-1, N, M, n) calcula el p-valor de la probabilidad de obtener al menos k éxitos en una muestra de tamaño n
                    p_val = stats.hypergeom.sf(k-1, N, M, n)
                    mean = n * (M / N)  # Media esperada bajo la hipótesis nula, la hipotesis nula es que el GO term no está enriquecido en el módulo
                    variance = n * (M / N) * (1 - M / N) * (N - n) / (N - 1)  # Varianza esperada bajo la hipótesis nula, se usa para calcular el z-score

                    # evitar division por cero
                    if variance > 0:
                        z_score = (k - mean) / np.sqrt(variance)
                    else:
                        z_score = 0.0

                p_values.append(p_val)
                z_scores.append(z_score)
                enriched_go_terms_list.append(go_term)
                
            if p_values:
                # corregir p.values usando FDR
                rejected, p_values_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh') # 0.05 es el nivel de significancia

                # Calcular el combined score para los términos significativos
                combined_scores = []
                for i in range(len(p_values_corrected)):
                    if rejected[i] and p_values_corrected[i] > 0 and z_scores[i] > 0:
                        score = -np.log10(p_values_corrected[i]) * z_scores[i]
                        combined_scores.append(score)
                    else:
                        combined_scores.append(0)
                
                # Seleccionar el GO term con el mayor combined score, este es el GO term representativo del módulo
                best_go_term = None
                max_combined_score = 0.0
                
                for i, go_term in enumerate(enriched_go_terms_list):
                    if combined_scores[i] > max_combined_score:
                        max_combined_score = combined_scores[i]
                        best_go_term = go_term
                
                if best_go_term:
                    go_term_name = go_term_details.loc[best_go_term, 'Term_Name_Clean'] if best_go_term in go_term_details.index else 'N/A'
                    representative_go = f"{best_go_term} ({go_term_name})"
                    # se usa el p-valor corregido del mejor GO term
                    representative_go_p_value = p_values_corrected[enriched_go_terms_list.index(best_go_term)]
                    representative_combined_score = max_combined_score # se guarda el score combinado del mejor GO term
                    representative_go_z_score = z_scores[enriched_go_terms_list.index(best_go_term)]
                else:
                    representative_go = "N/A (No GO terms enriched)"
                    representative_go_p_value = 1.0  # No se encontró GO enriquecido
                    representative_combined_score = 0.0  # No se encontró GO enriquecido
                    representative_go_z_score = 0.0  # No se encontró GO enriquecido

        # 2. Distribución de Proteínas DEG
        deg_counts = defaultdict(int)
        for p_id in module_proteins:
            deg_status = protein_deg_map.get(p_id, 'unknown')
            deg_counts[deg_status] += 1
        
        deg_distribution = {status: f"{(count / len(module_proteins) * 100):.2f}%" 
                            for status, count in deg_counts.items()}
        
        # 3. Distribución de Target_Group
        individual_t_counts = defaultdict(int)
        for p_id in module_proteins:
            target_group_str = protein_target_group_map.get(p_id, '')
            if target_group_str:
                for tg in target_group_str.split(','):
                    tg_stripped = tg.strip()
                    if tg_stripped:
                        individual_t_counts[tg_stripped] += 1
        
        total_proteins_in_module = len(module_proteins)
        target_group_distribution = {}
        for tg, count in individual_t_counts.items():
            if total_proteins_in_module > 0:
                target_group_distribution[tg] = f"{(count / total_proteins_in_module * 100):.2f}%"
            else:
                target_group_distribution[tg] = "0.00%"


        results.append({
            'module_id': module_id,
            'representative_go': representative_go,
            'representative_go_p_value': representative_go_p_value,
            'representative_combined_score': representative_combined_score,
            'representative_go_z_score': representative_go_z_score,
            'deg_distribution': deg_distribution,
            'target_group_distribution': target_group_distribution,
            'module_proteins': module_proteins
        })

    return results

## assign module go to proteins

In [ ]:
def assign_module_go_to_proteins(results):
    # a partir de los resultados de analyze_modules, asignar el GO representativo a cada proteína

    rows = []
    for module in results:
        # Excluir el módulo de ruido de esta asignación si no queremos asignarle un GO representativo 'N/A'
        if module['module_id'] == -1: 
            continue # No asigna GO representativo a proteínas de ruido aquí

        rep_go = module['representative_go']
        if '(' in rep_go and ')' in rep_go:
            go_term = rep_go.split('(')[0].strip()
            term_name = rep_go.split('(')[1].replace(')', '').strip()
        else:
            go_term = rep_go # si es "N/A (No GO terms enriched)" o similar, se asigna tal cual
            term_name = ""

        for prot in module['module_proteins']:
            rows.append({
                'proteina': prot,
                'GO_term': go_term,
                'Term_Name_Clean': term_name
            })
    return pd.DataFrame(rows)
print("✅ Funciones de análisis de módulos definidas.")

## Logic

In [ ]:
BASE_OUTPUT_DIR = './output' 
BASE_INPUT_DIR = './input'

# Parámetros del último HDBSCAN (debe coincidir con el nombre de su archivo .csv)
SUBDIR = './output/final_hdbscan_clusters' 
HDBSCAN_PARAMS_SUFFIX = "mcs20_ms20_cse0.00_alpha2.0" # <<< AJUSTAR ESTE VALOR >>>


# Rutas de Archivos de Entrada
EMBEDDINGS_PATH = os.path.join(BASE_OUTPUT_DIR, "normalized_embeddings.csv")
HDBSCAN_CLUSTER_LABELS_PATH = os.path.join(SUBDIR, f"hdbscan_cluster_labels_{HDBSCAN_PARAMS_SUFFIX}.csv")

# Rutas de Metadatos
go_path = os.path.join(BASE_INPUT_DIR, "GO.csv")
protein_metadata_path = os.path.join(BASE_INPUT_DIR, "metadata_proteins.csv")
go_metadata_path = os.path.join(BASE_INPUT_DIR, "metadata_GO.csv")

# Directorio de Salida para el Análisis de Módulos
FINAL_ANALYSIS_RESULTS_DIR = os.path.join(SUBDIR, "final_clusters_analysis")
os.makedirs(FINAL_ANALYSIS_RESULTS_DIR, exist_ok=True)

print(f"Ruta de Etiquetas HDBSCAN a cargar: {HDBSCAN_CLUSTER_LABELS_PATH}")
print(f"Directorio de Resultados de Análisis: {FINAL_ANALYSIS_RESULTS_DIR}")

In [ ]:
print("--- 1. Carga de Datos y Etiquetas ---")

# A. Cargar datos de anotación biológica
try:
    go_terms_df = pd.read_csv(go_path, sep='\t', dtype={'proteina': str}) 
    protein_metadata_df = pd.read_csv(protein_metadata_path, sep=',')
    go_metadata_df = pd.read_csv(go_metadata_path, sep=',')
    print("✅ Datos de anotación biológica (GO y Metadata) cargados y IDs convertidos a string.")
except Exception as e:
    print(f"🚨 ERROR al cargar archivos de anotación: {e}")
    sys.exit()

# B. Determinar el Universo de Proteínas (all_proteins_list)
all_proteins_list = []
try:
    df_embeddings = pd.read_csv(EMBEDDINGS_PATH)
    all_proteins_list = df_embeddings.iloc[:, 0].tolist()
    print(f"✅ Cargadas {len(all_proteins_list)} IDs de proteínas del universo de embeddings.")
except FileNotFoundError:
    print(f"⚠️ Advertencia: No se encontró {EMBEDDINGS_PATH}. Usando lista de proteínas de metadata_proteins.csv.")
    all_proteins_list = protein_metadata_df['proteina'].tolist()
    
# C. Cargar los resultados de clustering de HDBSCAN
try:
    df_hdbscan_labels = pd.read_csv(HDBSCAN_CLUSTER_LABELS_PATH)
    # Crear el diccionario protein_module_map
    protein_module_map = df_hdbscan_labels.set_index('protein_id')['cluster_label'].to_dict()
    print("✅ Etiquetas de clúster de HDBSCAN cargadas y mapeadas.")
    print(f"Ejemplo de mapeo: {list(protein_module_map.items())[:5]} (primeras 5)")
    print(f"Módulos únicos detectados: {np.unique(df_hdbscan_labels['cluster_label'].values)}")

except FileNotFoundError:
    print(f"🚨 ERROR: No se encontró el archivo de etiquetas de clúster en: {HDBSCAN_CLUSTER_LABELS_PATH}")
    sys.exit()
except Exception as e:
    print(f"🚨 ERROR al cargar o procesar las etiquetas de clúster de HDBSCAN: {e}")
    sys.exit()

In [ ]:
protein_go_map_diag = defaultdict(list)
for _, row in go_terms_df.iterrows():
    protein_go_map_diag[row['proteina']].append(row['GO_term'])

proteins_in_go_universe_diag = len(set(p for p in all_proteins_list if p in protein_go_map_diag))

print("\n--- Diagnóstico Inicial de Datos GO ---")
print(f"Total de proteínas en el universo de Embeddings: {len(all_proteins_list)}")
print(f"Total de proteínas en el Universo de GO (N): {proteins_in_go_universe_diag}")
if proteins_in_go_universe_diag == 0:
    print("🚨 ADVERTENCIA CRÍTICA: La población 'N' para el test GO es CERO. El problema de IDs persiste.")
# ----------------------------------------------------------------------

# --- Diagnóstico de Módulo 0 (Primer ID) ---
module_0_proteins = [p for p, m in protein_module_map.items() if m == 0]
if module_0_proteins:
    test_id = module_0_proteins[0]
    
    # Crear mapeos de metadatos (necesarios para el diagnóstico, aunque se hacen dentro de analyze_modules)
    protein_metadata_indexed = protein_metadata_df.set_index('proteina')
    
    deg_status = protein_metadata_indexed['DEG'].get(test_id, 'NO ENCONTRADO')
    target_group = protein_metadata_indexed['Target_group'].get(test_id, 'NO ENCONTRADO')
    go_terms = protein_go_map_diag.get(test_id, 'NO ENCONTRADO')
    
    print("\n--- Diagnóstico de Módulo 0 (Primer ID) ---")
    print(f"ID de prueba (Módulo 0): {test_id}")
    print(f"  Encontrado en metadata_proteins (DEG): {deg_status}")
    print(f"  Encontrado en metadata_proteins (Target): {target_group}")
    print(f"  Encontrado en GO.csv: {'Sí, con ' + str(len(go_terms)) + ' términos' if isinstance(go_terms, list) and go_terms else 'No/NO ENCONTRADO'}")
    print("------------------------------------------")

In [ ]:
print("\n--- 2. Fase: Análisis y Enriquecimiento de Módulos (HDBSCAN) ---")
    
# Ejecutar la función de análisis
analysis_results = analyze_modules(
    protein_module_map, go_terms_df, protein_metadata_df, go_metadata_df, all_proteins_list
)

print("\n--- Resultados del Análisis de Módulos Detectados (HDBSCAN) ---")
module_summary = []
for res in analysis_results:
    print(f"\n**Módulo {res['module_id']}** (Tamaño: {len(res['module_proteins'])} proteínas)") # Añadido el tamaño del módulo
    print(f"  Go Representativo: {res['representative_go']}")
    print(f"  p-value (corregido): {res['representative_go_p_value']:.2e}") # Más claro que es corregido
    print(f"  Combined score: {res['representative_combined_score']:.2f}") # Añadido el score combinado
    print(f"  Z-Score: {res['representative_go_z_score']:.2f}") # Añadido el Z-Score del GO representativo
        
    # Formatear DEG
    deg_str = ", ".join([f"{count}% {status}" for status, count in res['deg_distribution'].items()])
    print(f"  Proteínas DEG: {deg_str}")
        
    # Formatear Target Group
    target_group_str = ", ".join([f"{count}% {tg}" for tg, count in res['target_group_distribution'].items()])
    print(f"  Distribución de Target_Group: {target_group_str}")
    # La línea 'Total proteínas en módulo' ya está en la cabecera.
    # print(f"  Total proteínas en módulo: {len(res['module_proteins'])}") # Removida para evitar duplicidad

    # agregar a resumen
    module_summary.append({
        'module_id': res['module_id'],
        'module_size': len(res['module_proteins']), # Añadido el tamaño para el CSV
        'representative_go': res['representative_go'],
        'representative_go_p_value': res['representative_go_p_value'],
        'representative_combined_score': res['representative_combined_score'],
        'representative_go_z_score': res['representative_go_z_score'],
        'deg_distribution': str(res['deg_distribution']),
        'target_group_distribution': str(res['target_group_distribution']),
        'proteins': ", ".join(str(p) for p in res['module_proteins'])
        })



In [ ]:
# A. Guardar Resumen del Análisis de Módulos
results_df = pd.DataFrame(module_summary)
output_summary_path = os.path.join(FINAL_ANALYSIS_RESULTS_DIR, "hdbscan_module_analysis_summary.csv")
results_df.to_csv(output_summary_path, index=False)
print(f"✅ Resumen del análisis de módulos guardado en: {output_summary_path}")

# B. Asignar GO representativo a cada proteína y guardar
print("\n--- Generando lista de Nodos con Términos GO Más Representativos (por Módulo) ---")
protein_go_df = assign_module_go_to_proteins(analysis_results)

# Mostrar una muestra
print(protein_go_df.head().to_string(index=False))
if len(protein_go_df) > 5:
    print(f"... y {len(protein_go_df) - 5} más.")

# Guardar la tabla en archivo CSV
output_protein_go_path = os.path.join(FINAL_ANALYSIS_RESULTS_DIR, "hdbscan_proteins_with_representative_go.csv")
protein_go_df.to_csv(output_protein_go_path, index=False)
print(f"\n✅ Lista de nodos con GOs representativos guardada en: {output_protein_go_path}")

print("\n--- Proceso de Validación Biológica Completado ---")